# Cross-Industry Accelerator — Create Dashboards
### Auto-Create Real-Time KQL Dashboard + Power BI Report

This notebook creates **two dashboard types** for the selected industry:

| Type | Technology | Data Source | Use Case |
|---|---|---|---|
| **Real-Time Dashboard** | Fabric KQL Dashboard | Eventhouse | Live operational monitoring |
| **Static Report** | Power BI Report | Semantic Model / Warehouse | Historical analysis & KPIs |

**What it does:**
1. Reads industry configuration and auto-discovered schemas
2. Generates KQL queries for real-time tiles (event streams, anomalies, trends)
3. Creates a Fabric Real-Time Dashboard with auto-generated pages and tiles
4. Creates a Power BI Report shell linked to the semantic model

> **Prerequisites:**
> 1. Run `02_Load_Lakehouse.ipynb` for Eventhouse data (real-time dashboard)
> 2. Run `04_Create_Semantic_Model.ipynb` for the Power BI report
> 3. Eventhouse must be configured for real-time tiles

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AUTO-GENERATE KQL QUERIES FOR REAL-TIME TILES (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

import json, requests, base64
import sempy.fabric as fabric
import pandas as pd

# KQL reserved keywords that must be bracketed as ['keyword'] when used as column names
_KQL_RESERVED = {
    "and", "between", "by", "contains", "database", "date", "distinct", "evaluate",
    "extends", "false", "find", "has", "in", "let", "like", "limit", "not", "or",
    "order", "print", "project", "search", "set", "sort", "source", "summarize",
    "table", "time", "timestamp", "to", "top", "true", "type", "union", "where",
}

def _kql_col(name):
    """Bracket a column name if it's a KQL reserved keyword."""
    return f"['{name}']" if name.lower() in _KQL_RESERVED else name

# Build KQL queries from event/streaming table schemas
kql_tiles = []

for table_name in EVENTHOUSE_TABLES:
    # Read schema from Lakehouse Delta tables first, fall back to CSV
    kql_table = table_name.replace("fact_", "").replace("stream_", "")
    try:
        try:
            df = spark.read.table(table_name)
        except Exception:
            # Event/streaming tables only exist in Eventhouse, not as Delta tables
            # Fall back to reading schema from the CSV file
            csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
            df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

        # ZT: Validate KQL table name
        try:
            sanitize_identifier(kql_table)
        except ValueError as e:
            print(f"  ⚠ ZT: Skipping unsafe KQL table name: {kql_table!r}")
            log_audit_event("KQL_TILE_REJECTED", kql_table, str(e), "BLOCKED")
            continue

        # ZT: Validate all column names before interpolating into KQL queries
        all_cols = [f.name for f in df.schema.fields]
        safe_col_set = set(sanitize_columns(all_cols))

        cols = [c for c in all_cols if c in safe_col_set]

        # Detect timestamp and numeric columns for dashboarding (using validated names only)
        ts_cols = [c for c in cols if any(kw in c.lower() for kw in ["timestamp", "datetime", "date", "time", "created", "_at"])]
        num_cols = [f.name for f in df.schema.fields
                    if str(f.dataType) in ("IntegerType()", "LongType()", "FloatType()", "DoubleType()")
                    and not f.name.lower().endswith("_id")
                    and f.name in safe_col_set]
        cat_cols = [f.name for f in df.schema.fields
                    if str(f.dataType) == "StringType()" and not f.name.lower().endswith("_id")
                    and not any(kw in f.name.lower() for kw in ["name", "description", "notes", "comment"])
                    and f.name in safe_col_set]

        ts_col = ts_cols[0] if ts_cols else None

        # Tile 1: Event count time-series
        if ts_col:
            _ts = _kql_col(ts_col)
            kql_tiles.append({
                "title": f"{kql_table} — Events Over Time",
                "query": f"{kql_table}\n| summarize EventCount=count() by bin({_ts}, 1h)\n| order by {_ts} asc",
                "visualization": "timechart",
                "table": kql_table,
            })

        # Tile 2: Breakdown by category (top 10)
        if cat_cols:
            cat_col = cat_cols[0]
            _cat = _kql_col(cat_col)
            kql_tiles.append({
                "title": f"{kql_table} — By {cat_col}",
                "query": f"{kql_table}\n| summarize Count=count() by {_cat}\n| top 10 by Count desc",
                "visualization": "piechart",
                "table": kql_table,
            })

        # Tile 3: Numeric KPI (avg of first numeric column)
        if num_cols and ts_col:
            num_col = num_cols[0]
            _ts = _kql_col(ts_col)
            _num = _kql_col(num_col)
            label = num_col.replace("_", " ").title()
            kql_tiles.append({
                "title": f"{kql_table} — Avg {label} Trend",
                "query": f"{kql_table}\n| summarize Avg_{num_col}=avg({_num}) by bin({_ts}, 1h)\n| order by {_ts} asc",
                "visualization": "linechart",
                "table": kql_table,
            })
        _order_col = _kql_col(ts_col) if ts_col else _kql_col(cols[0])
        kql_tiles.append({
            "title": f"{kql_table} — Latest Events",
            "query": f"{kql_table}\n| top 20 by {_order_col} desc",
            "visualization": "table",
            "table": kql_table,
        })

        log_audit_event("KQL_TILES_GENERATED", kql_table,
            f"{sum(1 for t in kql_tiles if t['table'] == kql_table)} tiles")

    except Exception as e:
        print(f"  ⚠ Could not generate tiles for {table_name}: {e}")
        log_audit_event("KQL_TILE_ERROR", table_name, str(e), "ERROR")

print(f"✅ Generated {len(kql_tiles)} KQL dashboard tiles from {len(EVENTHOUSE_TABLES)} event tables.")
print(f"   (ZT: All column/table names validated before KQL interpolation)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE FABRIC REAL-TIME KQL DASHBOARD
# ════════════════════════════════════════════════════════════════════════
# Uses the correct KQL Dashboard definition format (schema_version 63)
# matching the Healthcare_Nursing_Dashboard.json pattern.

import uuid

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

RT_DASHBOARD_NAME = f"{INDUSTRY_LABEL}_RealTime_Dashboard"

if not EVENTHOUSE_CLUSTER_URI or not EVENTHOUSE_DATABASE:
    print("⚠ Eventhouse not configured — skipping real-time dashboard creation.")
else:
    # Resolve Eventhouse KQL database ID (needed for datasource definition)
    _eh_db_id = None
    _items = fabric.list_items()
    _kql_matches = _items[(_items["Type"] == "KQLDatabase") & (_items["Display Name"] == EVENTHOUSE_DATABASE)]
    if not _kql_matches.empty:
        _eh_db_id = str(_kql_matches.iloc[0]["Id"])
    print(f"  KQL Database ID: {_eh_db_id}")

    # Build datasource
    _ds_id = str(uuid.uuid4())
    datasource = {
        "kind": "kusto-trident",
        "scopeId": "kusto-trident",
        "clusterUri": EVENTHOUSE_CLUSTER_URI,
        "database": _eh_db_id or EVENTHOUSE_DATABASE,
        "name": EVENTHOUSE_DATABASE,
        "id": _ds_id,
        "workspace": workspace_id,
    }

    # Group tiles by table → pages
    pages_by_table = {}
    for tile in kql_tiles:
        tbl = tile["table"]
        if tbl not in pages_by_table:
            pages_by_table[tbl] = []
        pages_by_table[tbl].append(tile)

    # Build pages, queries, and tiles arrays
    pages = []
    queries = []
    tiles = []

    for page_idx, (tbl_name, page_tiles) in enumerate(pages_by_table.items()):
        page_id = str(uuid.uuid4())
        page_display = tbl_name.replace("_", " ").title()
        pages.append({"id": page_id, "name": page_display})

        for tile_idx, tile in enumerate(page_tiles):
            query_id = str(uuid.uuid4())
            tile_id = str(uuid.uuid4())

            queries.append({
                "dataSource": {"kind": "inline", "dataSourceId": _ds_id},
                "text": tile["query"],
                "id": query_id,
                "usedVariables": [],
            })

            # Map visualization types
            vis_map = {
                "timechart": "timechart",
                "linechart": "timechart",
                "piechart": "pie",
                "table": "table",
                "barchart": "bar",
                "columnchart": "column",
            }
            vis_type = vis_map.get(tile["visualization"], tile["visualization"])

            tiles.append({
                "id": tile_id,
                "title": tile["title"],
                "visualType": vis_type,
                "pageId": page_id,
                "layout": {
                    "x": (tile_idx % 2) * 6,
                    "y": (tile_idx // 2) * 8,
                    "width": 12 if vis_type == "table" else 6,
                    "height": 8,
                },
                "queryRef": {"kind": "query", "queryId": query_id},
                "visualOptions": {
                    "multipleYAxes": {
                        "base": {"id": "-1", "columns": [], "label": "",
                                 "yAxisMinimumValue": None, "yAxisMaximumValue": None,
                                 "yAxisScale": "linear", "horizontalLines": []},
                        "additional": [], "showMultiplePanels": False,
                    },
                    "hideLegend": False, "legendLocation": "bottom",
                    "xColumnTitle": "", "xColumn": None, "yColumns": None,
                    "seriesColumns": None, "xAxisScale": "linear", "verticalLine": "",
                    "crossFilterDisabled": True, "drillthroughDisabled": False,
                    "crossFilter": [], "drillthrough": [],
                },
            })

    # Assemble full dashboard definition
    dashboard_definition = {
        "schema_version": 63,
        "title": f"{INDUSTRY_LABEL} Real-Time Operations",
        "dataSources": [datasource],
        "pages": pages,
        "parameters": [],
        "baseQueries": [],
        "queries": queries,
        "tiles": tiles,
    }

    # Delete existing dashboard if present
    _existing = _items[(_items["Display Name"] == RT_DASHBOARD_NAME)]
    if not _existing.empty:
        _eid = str(_existing.iloc[0]["Id"])
        requests.delete(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{_eid}",
            headers={"Authorization": f"Bearer {access_token}"}, timeout=30
        )
        print(f"  Deleted existing dashboard: {_eid}")
        import time; time.sleep(30)  # Wait for name to become available

    # Create via REST API
    FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}

    dash_b64 = base64.b64encode(json.dumps(dashboard_definition).encode("utf-8")).decode("utf-8")

    payload = {
        "displayName": RT_DASHBOARD_NAME,
        "description": f"Real-time operational dashboard for {INDUSTRY}",
        "type": "KQLDashboard",
        "definition": {
            "parts": [{
                "path": "RealTimeDashboard.json",
                "payload": dash_b64,
                "payloadType": "InlineBase64"
            }]
        }
    }

    print(f"Creating real-time dashboard: {RT_DASHBOARD_NAME}...")
    import time as _time
    for _attempt in range(5):
        resp = requests.post(url, headers=headers, json=payload)
        if resp.status_code in (200, 201, 202):
            break
        if "NotAvailableYet" in resp.text:
            print(f"  Name not available yet, retrying in 25s ({_attempt+1}/5)...")
            _time.sleep(25)
        else:
            break

    if resp.status_code in (200, 201, 202):
        print(f"\n✅ Real-Time Dashboard created: {RT_DASHBOARD_NAME}")
        resp_json = resp.json()
        if "id" in resp_json:
            print(f"   Item ID: {resp_json['id']}")
        print(f"   Pages:   {len(pages)}")
        print(f"   Queries: {len(queries)}")
        print(f"   Tiles:   {len(tiles)}")
    else:
        print(f"\n⚠ Dashboard creation returned status {resp.status_code}")
        print(f"   Response: {resp.text[:500]}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE POWER BI REPORT (STATIC DASHBOARD) — WITH AUTO-GENERATED VISUALS
# ════════════════════════════════════════════════════════════════════════
# Uses the PBIR-Legacy format with auto-generated visuals for each page:
#   Executive Summary: KPI cards + charts from headline tables
#   Per-table pages:   KPI row → chart row → donut + detail table

import uuid as _uuid

REPORT_NAME = f"{INDUSTRY_LABEL}_Analytics_Report"

# ── PBI visual container helper functions ──────────────────────────────

def _uid():
    return str(_uuid.uuid4()).replace("-", "")[:16]

def _src(alias):
    return {"SourceRef": {"Source": alias}}

def _col_expr(alias, prop):
    return {"Column": {"Expression": _src(alias), "Property": prop}}

def _col_select(alias, prop):
    return {"Column": {"Expression": _src(alias), "Property": prop}, "Name": f"{alias}.{prop}"}

def _agg_select(alias, prop, func):
    """func: 0=Sum, 1=Avg, 2=Min, 3=Max, 4=CountNonNull, 5=Count"""
    fname = {0: "Sum", 1: "Avg", 2: "Min", 3: "Max", 4: "CountNonNull", 5: "Count"}[func]
    return {
        "Aggregation": {
            "Expression": _col_expr(alias, prop),
            "Function": func
        },
        "Name": f"{fname}({alias}.{prop})"
    }

def _best_agg(col_name):
    """Pick Sum(0) vs Avg(1) based on column name semantics."""
    low = col_name.lower()
    if any(kw in low for kw in ["pct", "score", "rate", "ratio", "accuracy",
                                 "likelihood", "balance", "satisfaction", "avg",
                                 "occupancy", "fatigue", "burden", "pressure"]):
        return 1  # Avg
    return 0  # Sum

def _metric_label(col_name, agg_func):
    prefix = "Avg" if agg_func == 1 else "Total"
    label = col_name.replace("_", " ").title()
    return f"{prefix} {label}"

def _get_numeric_cols(df_schema):
    results = []
    for f in df_schema.fields:
        if str(f.dataType) in ("IntegerType()", "LongType()", "FloatType()", "DoubleType()"):
            if not f.name.lower().endswith("_id"):
                results.append((f.name, _best_agg(f.name)))
    return results

def _title_obj(title_text):
    """Standard title + subtitle vcObjects block."""
    return {
        "title": [{"properties": {
            "show": {"expr": {"Literal": {"Value": "true"}}},
            "titleWrap": {"expr": {"Literal": {"Value": "true"}}},
            "text": {"expr": {"Literal": {"Value": json.dumps(title_text)}}}
        }}],
    }

def _make_card(x, y, w, h, table_name, col_name, agg_func=0, title_text="",
               font_color=None, bg_color=None):
    a = "t"
    name = _uid()
    sel = _agg_select(a, col_name, agg_func)
    frm = [{"Name": a, "Entity": table_name, "Type": 0}]
    vc_objs = {}
    if title_text:
        vc_objs.update(_title_obj(title_text))
    # Card label styling
    vc_objs["labels"] = [{"properties": {
        "fontSize": {"expr": {"Literal": {"Value": "28D"}}},
    }}]
    if font_color:
        vc_objs["labels"][0]["properties"]["color"] = {
            "solid": {"color": {"expr": {"Literal": {"Value": json.dumps(font_color)}}}}
        }
    if bg_color:
        vc_objs["background"] = [{"properties": {
            "show": {"expr": {"Literal": {"Value": "true"}}},
            "color": {"solid": {"color": {"expr": {"Literal": {"Value": json.dumps(bg_color)}}}}}
        }}]
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "width": w, "height": h, "z": 0, "tabOrder": 0}}],
        "singleVisual": {
            "visualType": "card",
            "projections": {"Values": [{"queryRef": sel["Name"]}]},
            "prototypeQuery": {"Version": 2, "From": frm, "Select": [sel]},
            "vcObjects": vc_objs,
        },
    }
    query = {"Commands": [{"SemanticQueryDataShapeCommand": {
        "Query": {"Version": 2, "From": frm, "Select": [sel]},
        "Binding": {"Primary": {"Groupings": [{"Projections": [0]}]},
                    "DataReduction": {"DataVolume": 4, "Primary": {"Top": {"Count": 1}}}},
        "ExecutionMetricsKind": 1
    }}]}
    return {"x": x, "y": y, "z": 0, "width": w, "height": h,
            "config": json.dumps(config), "filters": "[]",
            "query": json.dumps(query),
            "dataTransforms": json.dumps({"projectionOrdering": {"Values": [0]}})}

def _make_column_chart(x, y, w, h, table_name, cat_col, val_col, agg_func=0, title_text=""):
    a = "t"
    name = _uid()
    cat_sel = _col_select(a, cat_col)
    val_sel = _agg_select(a, val_col, agg_func)
    frm = [{"Name": a, "Entity": table_name, "Type": 0}]
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "width": w, "height": h, "z": 0, "tabOrder": 0}}],
        "singleVisual": {
            "visualType": "clusteredColumnChart",
            "projections": {"Category": [{"queryRef": cat_sel["Name"]}],
                            "Y": [{"queryRef": val_sel["Name"]}]},
            "prototypeQuery": {"Version": 2, "From": frm, "Select": [cat_sel, val_sel]},
            "vcObjects": _title_obj(title_text) if title_text else {},
        },
    }
    query = {"Commands": [{"SemanticQueryDataShapeCommand": {
        "Query": {"Version": 2, "From": frm, "Select": [cat_sel, val_sel]},
        "Binding": {"Primary": {"Groupings": [{"Projections": [0]}]},
                    "Values": [{"Projections": [1]}],
                    "DataReduction": {"DataVolume": 4, "Primary": {"Top": {"Count": 1000}}}},
        "ExecutionMetricsKind": 1
    }}]}
    return {"x": x, "y": y, "z": 0, "width": w, "height": h,
            "config": json.dumps(config), "filters": "[]",
            "query": json.dumps(query),
            "dataTransforms": json.dumps({"projectionOrdering": {"Category": [0], "Y": [1]}})}

def _make_bar_chart(x, y, w, h, table_name, cat_col, val_col, agg_func=0, title_text=""):
    """Horizontal bar chart — good for ranked categories."""
    a = "t"
    name = _uid()
    cat_sel = _col_select(a, cat_col)
    val_sel = _agg_select(a, val_col, agg_func)
    frm = [{"Name": a, "Entity": table_name, "Type": 0}]
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "width": w, "height": h, "z": 0, "tabOrder": 0}}],
        "singleVisual": {
            "visualType": "barChart",
            "projections": {"Category": [{"queryRef": cat_sel["Name"]}],
                            "Y": [{"queryRef": val_sel["Name"]}]},
            "prototypeQuery": {"Version": 2, "From": frm, "Select": [cat_sel, val_sel]},
            "vcObjects": _title_obj(title_text) if title_text else {},
        },
    }
    query = {"Commands": [{"SemanticQueryDataShapeCommand": {
        "Query": {"Version": 2, "From": frm, "Select": [cat_sel, val_sel]},
        "Binding": {"Primary": {"Groupings": [{"Projections": [0]}]},
                    "Values": [{"Projections": [1]}],
                    "DataReduction": {"DataVolume": 4, "Primary": {"Top": {"Count": 1000}}}},
        "ExecutionMetricsKind": 1
    }}]}
    return {"x": x, "y": y, "z": 0, "width": w, "height": h,
            "config": json.dumps(config), "filters": "[]",
            "query": json.dumps(query),
            "dataTransforms": json.dumps({"projectionOrdering": {"Category": [0], "Y": [1]}})}

def _make_donut_chart(x, y, w, h, table_name, cat_col, val_col, agg_func=0, title_text=""):
    a = "t"
    name = _uid()
    cat_sel = _col_select(a, cat_col)
    val_sel = _agg_select(a, val_col, agg_func)
    frm = [{"Name": a, "Entity": table_name, "Type": 0}]
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "width": w, "height": h, "z": 0, "tabOrder": 0}}],
        "singleVisual": {
            "visualType": "donutChart",
            "projections": {"Category": [{"queryRef": cat_sel["Name"]}],
                            "Y": [{"queryRef": val_sel["Name"]}]},
            "prototypeQuery": {"Version": 2, "From": frm, "Select": [cat_sel, val_sel]},
            "vcObjects": _title_obj(title_text) if title_text else {},
        },
    }
    query = {"Commands": [{"SemanticQueryDataShapeCommand": {
        "Query": {"Version": 2, "From": frm, "Select": [cat_sel, val_sel]},
        "Binding": {"Primary": {"Groupings": [{"Projections": [0]}]},
                    "Values": [{"Projections": [1]}],
                    "DataReduction": {"DataVolume": 4, "Primary": {"Top": {"Count": 100}}}},
        "ExecutionMetricsKind": 1
    }}]}
    return {"x": x, "y": y, "z": 0, "width": w, "height": h,
            "config": json.dumps(config), "filters": "[]",
            "query": json.dumps(query),
            "dataTransforms": json.dumps({"projectionOrdering": {"Category": [0], "Y": [1]}})}

def _make_line_chart(x, y, w, h, table_name, date_col, val_col, agg_func=0, title_text=""):
    a = "t"
    name = _uid()
    date_sel = _col_select(a, date_col)
    val_sel = _agg_select(a, val_col, agg_func)
    frm = [{"Name": a, "Entity": table_name, "Type": 0}]
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "width": w, "height": h, "z": 0, "tabOrder": 0}}],
        "singleVisual": {
            "visualType": "lineChart",
            "projections": {"Category": [{"queryRef": date_sel["Name"]}],
                            "Y": [{"queryRef": val_sel["Name"]}]},
            "prototypeQuery": {"Version": 2, "From": frm, "Select": [date_sel, val_sel]},
            "vcObjects": _title_obj(title_text) if title_text else {},
        },
    }
    query = {"Commands": [{"SemanticQueryDataShapeCommand": {
        "Query": {"Version": 2, "From": frm, "Select": [date_sel, val_sel]},
        "Binding": {"Primary": {"Groupings": [{"Projections": [0]}]},
                    "Values": [{"Projections": [1]}],
                    "DataReduction": {"DataVolume": 4, "Primary": {"Window": {"Count": 1000}}}},
        "ExecutionMetricsKind": 1
    }}]}
    return {"x": x, "y": y, "z": 0, "width": w, "height": h,
            "config": json.dumps(config), "filters": "[]",
            "query": json.dumps(query),
            "dataTransforms": json.dumps({"projectionOrdering": {"Category": [0], "Y": [1]}})}

def _make_table_visual(x, y, w, h, table_name, columns, title_text=""):
    a = "t"
    name = _uid()
    sels = [_col_select(a, c) for c in columns]
    frm = [{"Name": a, "Entity": table_name, "Type": 0}]
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "width": w, "height": h, "z": 0, "tabOrder": 0}}],
        "singleVisual": {
            "visualType": "tableEx",
            "projections": {"Values": [{"queryRef": s["Name"]} for s in sels]},
            "prototypeQuery": {"Version": 2, "From": frm, "Select": sels},
            "vcObjects": _title_obj(title_text) if title_text else {},
        },
    }
    query = {"Commands": [{"SemanticQueryDataShapeCommand": {
        "Query": {"Version": 2, "From": frm, "Select": sels},
        "Binding": {"Primary": {"Groupings": [{"Projections": list(range(len(sels)))}]},
                    "DataReduction": {"DataVolume": 4, "Primary": {"Window": {"Count": 500}}}},
        "ExecutionMetricsKind": 1
    }}]}
    return {"x": x, "y": y, "z": 0, "width": w, "height": h,
            "config": json.dumps(config), "filters": "[]",
            "query": json.dumps(query),
            "dataTransforms": json.dumps({"projectionOrdering": {"Values": list(range(len(sels)))}})}

# ── Resolve semantic model ─────────────────────────────────────────────
items_df = fabric.list_items()
sm_matches = items_df[(items_df["Type"] == "SemanticModel") & (items_df["Display Name"] == SEMANTIC_MODEL_NAME)]

if sm_matches.empty:
    print(f"⚠ Semantic model '{SEMANTIC_MODEL_NAME}' not found.")
    print(f"   Run 04_Create_Semantic_Model.ipynb first.")
    print(items_df[items_df["Type"] == "SemanticModel"][["Display Name", "Id"]].to_string(index=False))
else:
    sm_id = str(sm_matches.iloc[0].Id)
    print(f"✓ Semantic model: {SEMANTIC_MODEL_NAME} → {sm_id}")

    # ── Part 1: definition.pbir ──
    pbir_definition = {
        "version": "1.0",
        "datasetReference": {
            "byConnection": {
                "connectionString": None,
                "pbiServiceModelId": None,
                "pbiModelVirtualServerName": "sobe_wowvirtualserver",
                "pbiModelDatabaseName": sm_id,
                "name": "EntityDataSource",
                "connectionType": "pbiServiceXmlaStyleLive"
            }
        }
    }

    # ── Pre-scan all tables for schema info ────────────────────────────
    fact_tables_all = FACT_TABLES_BATCH + FACT_TABLES_EVENT
    _table_info = {}  # table_name → {metrics, ts_col, cat_col, all_cols}
    _ACCENT = ["#118DFF", "#12239E", "#E66C37", "#6B007B", "#E044A7", "#744EC2",
               "#D9B300", "#D64550", "#197278", "#1AAB40", "#36A5A5", "#A85E00"]

    for ft in fact_tables_all:
        try:
            try:
                _df = spark.read.table(ft)
            except Exception:
                _df = spark.read.option("header", True).option("inferSchema", True).csv(f"{_FS_CSV_PATH}/{ft}.csv")
            all_cols = [f.name for f in _df.schema.fields]
            metrics = _get_numeric_cols(_df.schema)
            ts_cols = [c for c in all_cols
                       if any(kw in c.lower() for kw in ["timestamp", "datetime", "date", "time", "created", "_at"])]
            cat_cols = [f.name for f in _df.schema.fields
                        if str(f.dataType) == "StringType()" and not f.name.lower().endswith("_id")
                        and not any(kw in f.name.lower() for kw in ["name", "description", "notes", "comment"])]
            _table_info[ft] = {
                "metrics": metrics, "ts_col": ts_cols[0] if ts_cols else None,
                "cat_col": cat_cols[0] if cat_cols else None, "all_cols": all_cols,
            }
        except Exception as e:
            print(f"  ⚠ Schema read failed for {ft}: {e}")

    sections = []

    # ═══════════════════════════════════════════════════════════════════
    # EXECUTIVE SUMMARY — KPI cards + headline charts
    # Layout: Row 1 = 4 colored KPI cards
    #         Row 2 = column chart (left) + donut (right)
    #         Row 3 = line chart (left) + bar chart (right)
    # ═══════════════════════════════════════════════════════════════════
    exec_visuals = []

    # Pick the 4 most impactful metrics across all tables
    _top_kpis = []
    for ft in fact_tables_all[:6]:
        info = _table_info.get(ft)
        if not info or not info["metrics"]:
            continue
        table_label = ft.replace("fact_", "").replace("_", " ").title()
        for col_name, agg_func in info["metrics"][:1]:  # best metric per table
            _top_kpis.append((ft, col_name, agg_func, table_label))
    _top_kpis = _top_kpis[:4]

    # Row 1: 4 colored KPI cards
    for i, (ft, col_name, agg_func, table_label) in enumerate(_top_kpis):
        color = _ACCENT[i % len(_ACCENT)]
        exec_visuals.append(
            _make_card(20 + i * 310, 10, 290, 120, ft, col_name, agg_func=agg_func,
                       title_text=f"{table_label}", font_color=color)
        )

    # Row 2: Column chart + Donut
    # Find first table with a category column for the column chart
    _chart_ft = None
    for ft in fact_tables_all[:6]:
        info = _table_info.get(ft)
        if info and info["cat_col"] and info["metrics"]:
            _chart_ft = ft
            break
    if _chart_ft:
        info = _table_info[_chart_ft]
        m_col, m_agg = info["metrics"][0]
        tbl_label = _chart_ft.replace("fact_", "").replace("_", " ").title()
        exec_visuals.append(
            _make_column_chart(20, 145, 620, 270, _chart_ft, info["cat_col"], m_col, m_agg,
                               title_text=f"{tbl_label}: {_metric_label(m_col, m_agg)} by {info['cat_col'].replace('_', ' ').title()}")
        )
    # Donut from a different table
    _donut_ft = None
    for ft in fact_tables_all[:6]:
        info = _table_info.get(ft)
        if info and info["cat_col"] and info["metrics"] and ft != _chart_ft:
            _donut_ft = ft
            break
    if _donut_ft:
        info = _table_info[_donut_ft]
        m_col, m_agg = info["metrics"][0]
        tbl_label = _donut_ft.replace("fact_", "").replace("_", " ").title()
        exec_visuals.append(
            _make_donut_chart(660, 145, 600, 270, _donut_ft, info["cat_col"], m_col, m_agg,
                              title_text=f"{tbl_label}: {info['cat_col'].replace('_', ' ').title()} Distribution")
        )

    # Row 3: Line chart + Bar chart
    _line_ft = None
    for ft in fact_tables_all[:6]:
        info = _table_info.get(ft)
        if info and info["ts_col"] and info["metrics"]:
            _line_ft = ft
            break
    if _line_ft:
        info = _table_info[_line_ft]
        m_col, m_agg = info["metrics"][0]
        tbl_label = _line_ft.replace("fact_", "").replace("_", " ").title()
        exec_visuals.append(
            _make_line_chart(20, 430, 620, 275, _line_ft, info["ts_col"], m_col, m_agg,
                             title_text=f"{tbl_label}: {_metric_label(m_col, m_agg)} Trend")
        )
    _bar_ft = None
    for ft in fact_tables_all[:6]:
        info = _table_info.get(ft)
        if info and info["cat_col"] and len(info["metrics"]) > 1 and ft != _chart_ft:
            _bar_ft = ft
            break
    if _bar_ft:
        info = _table_info[_bar_ft]
        m_col, m_agg = info["metrics"][1] if len(info["metrics"]) > 1 else info["metrics"][0]
        tbl_label = _bar_ft.replace("fact_", "").replace("_", " ").title()
        exec_visuals.append(
            _make_bar_chart(660, 430, 600, 275, _bar_ft, info["cat_col"], m_col, m_agg,
                            title_text=f"{tbl_label}: {_metric_label(m_col, m_agg)} by {info['cat_col'].replace('_', ' ').title()}")
        )

    print(f"  Executive Summary: {len(exec_visuals)} visuals (cards + charts)")

    sections.append({
        "name": "SectionExecSummary",
        "displayName": "Executive Summary",
        "filters": "[]",
        "ordinal": 0,
        "visualContainers": exec_visuals,
        "config": json.dumps({"layouts": [{"id": 0, "position": {}}]}),
        "displayOption": 1,
        "width": 1280,
        "height": 720,
    })

    # ═══════════════════════════════════════════════════════════════════
    # PER-TABLE PAGES
    # Layout: Row 1 (y=10,  h=100):  up to 3 KPI cards
    #         Row 2 (y=120, h=250):  column chart (left) + line chart (right)
    #         Row 3 (y=385, h=320):  donut (left, w=380) + detail table (right)
    # ═══════════════════════════════════════════════════════════════════
    for pg_idx, ft in enumerate(fact_tables_all[:10]):
        label = ft.replace("fact_", "").replace("_", " ").title()
        visuals = []
        info = _table_info.get(ft)
        if not info:
            sections.append({
                "name": f"Section{pg_idx}", "displayName": label,
                "filters": "[]", "ordinal": pg_idx + 1, "visualContainers": [],
                "config": json.dumps({"layouts": [{"id": 0, "position": {}}]}),
                "displayOption": 1, "width": 1280, "height": 720,
            })
            continue

        metrics = info["metrics"]
        ts_col = info["ts_col"]
        cat_col = info["cat_col"]
        all_cols = info["all_cols"]

        # Row 1: Up to 3 KPI Cards (colored)
        n_cards = min(len(metrics), 3)
        card_w = (1240 // max(n_cards, 1)) - 10
        for c_idx, (col_name, agg_func) in enumerate(metrics[:3]):
            color = _ACCENT[c_idx % len(_ACCENT)]
            visuals.append(
                _make_card(20 + c_idx * (card_w + 10), 10, card_w, 100, ft,
                           col_name, agg_func=agg_func,
                           title_text=_metric_label(col_name, agg_func),
                           font_color=color)
            )

        # Row 2: Column chart (left) + Line chart (right)
        if cat_col and metrics:
            m_col, m_agg = metrics[0]
            visuals.append(
                _make_column_chart(20, 125, 610, 245, ft, cat_col, m_col, m_agg,
                                   title_text=f"{_metric_label(m_col, m_agg)} by {cat_col.replace('_', ' ').title()}")
            )
        if ts_col and metrics:
            m_col, m_agg = metrics[0]
            visuals.append(
                _make_line_chart(650, 125, 610, 245, ft, ts_col, m_col, m_agg,
                                 title_text=f"{_metric_label(m_col, m_agg)} Over Time")
            )

        # Row 3: Donut (left) + Table (right)
        if cat_col and len(metrics) > 1:
            m_col, m_agg = metrics[1]
            visuals.append(
                _make_donut_chart(20, 385, 380, 320, ft, cat_col, m_col, m_agg,
                                  title_text=f"{_metric_label(m_col, m_agg)} Distribution")
            )
            # Table beside the donut
            show_cols = all_cols[:7]
            visuals.append(
                _make_table_visual(420, 385, 840, 320, ft, show_cols,
                                    title_text=f"{label} — Detail")
            )
        else:
            # No donut → full-width table
            show_cols = all_cols[:8]
            visuals.append(
                _make_table_visual(20, 385, 1240, 320, ft, show_cols,
                                    title_text=f"{label} — Detail")
            )

        _vis_types = [json.loads(v["config"]).get("singleVisual", {}).get("visualType", "?") for v in visuals]
        print(f"  {label}: {len(visuals)} visuals → {', '.join(_vis_types)}")

        sections.append({
            "name": f"Section{pg_idx}",
            "displayName": label,
            "filters": "[]",
            "ordinal": pg_idx + 1,
            "visualContainers": visuals,
            "config": json.dumps({"layouts": [{"id": 0, "position": {}}]}),
            "displayOption": 1,
            "width": 1280,
            "height": 720,
        })

    # ── Assemble report.json ──
    report_json = {
        "id": "0",
        "resourcePackages": [{
            "resourcePackage": {
                "name": "SharedResources",
                "type": 2,
                "items": [{"type": 202, "path": "BaseThemes/CY24SU06.json", "name": "CY24SU06"}],
                "disabled": False
            }
        }],
        "sections": sections,
        "config": json.dumps({
            "version": "5.50",
            "themeCollection": {"baseTheme": {"name": "CY24SU06", "version": "5.50", "type": 2}},
            "activeSectionIndex": 0,
            "defaultDrillFilterOtherVisuals": True,
            "linguisticSchemaSyncVersion": 2,
            "settings": {"useDefaultAggregateDisplayName": True, "useStylableVisualContainerHeader": True}
        }),
        "layoutOptimization": 0,
        "publicCustomVisuals": [],
    }

    total_visuals = sum(len(s["visualContainers"]) for s in sections)
    print(f"\n  Report structure: {len(sections)} pages, {total_visuals} visuals total")

    pbir_b64 = base64.b64encode(json.dumps(pbir_definition).encode("utf-8")).decode("utf-8")
    report_b64 = base64.b64encode(json.dumps(report_json).encode("utf-8")).decode("utf-8")

    # Refresh token (may have expired during visual generation)
    access_token = notebookutils.credentials.getToken('pbi')

    # Delete existing report if present
    items_df = fabric.list_items()
    _existing_rpt = items_df[(items_df["Display Name"] == REPORT_NAME)]
    if not _existing_rpt.empty:
        _rpt_id = str(_existing_rpt.iloc[0]["Id"])
        requests.delete(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{_rpt_id}",
            headers={"Authorization": f"Bearer {access_token}"}, timeout=30
        )
        print(f"  Deleted existing report: {_rpt_id}")
        import time; time.sleep(15)

    payload = {
        "displayName": REPORT_NAME,
        "description": f"Analytics report for {INDUSTRY} — auto-generated visuals linked to {SEMANTIC_MODEL_NAME}",
        "type": "Report",
        "definition": {
            "parts": [
                {"path": "definition.pbir", "payload": pbir_b64, "payloadType": "InlineBase64"},
                {"path": "report.json", "payload": report_b64, "payloadType": "InlineBase64"}
            ]
        }
    }

    _rpt_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"
    _rpt_headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}

    print(f"\nCreating Power BI report: {REPORT_NAME}...")
    import time as _time
    for _attempt in range(5):
        resp = requests.post(_rpt_url, headers=_rpt_headers, json=payload)
        if resp.status_code in (200, 201, 202):
            break
        if "NotAvailableYet" in resp.text:
            print(f"  Name not available yet, retrying in 15s ({_attempt+1}/5)...")
            _time.sleep(15)
        else:
            break

    if resp.status_code in (200, 201, 202):
        print(f"\n✅ Report created: {REPORT_NAME}")
        try:
            resp_json = resp.json()
        except Exception:
            resp_json = None
        if resp_json and "id" in resp_json:
            print(f"   Item ID: {resp_json['id']}")
        print(f"   Pages:   {len(sections)}")
        print(f"   Visuals: {total_visuals}")
        print(f"   Linked to: {SEMANTIC_MODEL_NAME}")
        print(f"\n   Visual types per page:")
        for s in sections:
            vc = len(s["visualContainers"])
            print(f"     • {s['displayName']}: {vc} visual{'s' if vc != 1 else ''}")
    else:
        print(f"\n⚠ Report creation returned status {resp.status_code}")
        print(f"   Response: {resp.text[:500]}")
        print(f"\n   You can create the report manually in Power BI using the semantic model.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DASHBOARD CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*65}")
print(f"DASHBOARD SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*65}")
print(f"")
print(f"  Real-Time Dashboard:  {RT_DASHBOARD_NAME if EVENTHOUSE_CLUSTER_URI else 'N/A (Eventhouse not configured)'}")
if EVENTHOUSE_CLUSTER_URI:
    print(f"    Data Source:        {EVENTHOUSE_CLUSTER_URI}")
    print(f"    Database:           {EVENTHOUSE_DATABASE}")
    print(f"    Pages:              {len(pages_by_table) if 'pages_by_table' in dir() else 0}")
    print(f"    KQL Tiles:          {len(kql_tiles)}")
    print(f"    Auto-Refresh:       30 seconds")
print(f"")
print(f"  Power BI Report:      {REPORT_NAME if 'REPORT_NAME' in dir() else 'N/A'}")
if 'sections' in dir():
    print(f"    Semantic Model:     {SEMANTIC_MODEL_NAME}")
    print(f"    Pages:              {len(sections)}")
print(f"")
print(f"{'═'*65}")
print(f"✅ All dashboards created. The {INDUSTRY} accelerator pipeline is complete!")
print(f"")
print(f"  Pipeline Summary:")
print(f"  00 → Industry Config")
print(f"  01 → Data Ingestion & Discovery")
print(f"  02 → Lakehouse + Eventhouse Loading")
print(f"  03 → Warehouse Loading")
print(f"  04 → Semantic Model Creation")
print(f"  05 → Ontology Creation")
print(f"  06 → Data Agent Creation")
print(f"  07 → Dashboard Creation ← YOU ARE HERE")